## ML_1021: Sliding Window Attention

**Difficulty**: Hard | **TorchCode**: #11

### Problem
Implement Sliding Window Attention (Mistral-style). Position `i` can only attend to positions `j` where `|i - j| ≤ window_size`. Mask all other positions with `-inf`.

### Formula
$$M_{ij} = \begin{cases} 0 & |i-j| \le w \\ -\infty & |i-j| > w \end{cases}$$

In [ ]:
import torch
import math

def sliding_window_attention(Q, K, V, window_size):
    d_k = K.size(-1)
    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)
    S = Q.size(1)
    idx = torch.arange(S, device=Q.device)
    mask = (idx.unsqueeze(0) - idx.unsqueeze(1)).abs() > window_size
    scores = scores.masked_fill(mask.unsqueeze(0), float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return torch.bmm(weights, V)

In [ ]:
import torch
import math

# --- Test 1: Output shape ---
out = sliding_window_attention(torch.randn(2, 8, 16), torch.randn(2, 8, 16), torch.randn(2, 8, 16), window_size=2)
assert out.shape == (2, 8, 16)

# --- Test 2: window_size=0 — only sees itself ---
Q = torch.randn(1, 4, 8); K = torch.randn(1, 4, 8); V = torch.randn(1, 4, 8)
out = sliding_window_attention(Q, K, V, window_size=0)
assert torch.allclose(out, V, atol=1e-5)

# --- Test 3: Large window equals full attention ---
torch.manual_seed(0)
B, S, D = 2, 6, 8
Q = torch.randn(B, S, D); K = torch.randn(B, S, D); V = torch.randn(B, S, D)
out_win = sliding_window_attention(Q, K, V, window_size=S)
scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(D)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
assert torch.allclose(out_win, ref, atol=1e-5)

# --- Test 4: Distant tokens don't affect output ---
torch.manual_seed(0)
B, S, D = 1, 10, 8
Q = torch.randn(B, S, D); K = torch.randn(B, S, D); V = torch.randn(B, S, D)
out1 = sliding_window_attention(Q, K, V, window_size=1)
K2, V2 = K.clone(), V.clone()
K2[:, 5:] = torch.randn(B, 5, D); V2[:, 5:] = torch.randn(B, 5, D)
out2 = sliding_window_attention(Q, K2, V2, window_size=1)
assert torch.allclose(out1[:, 0], out2[:, 0], atol=1e-5)

# --- Test 5: Gradient flow ---
Q = torch.randn(2, 4, 8, requires_grad=True)
K = torch.randn(2, 4, 8, requires_grad=True)
V = torch.randn(2, 4, 8, requires_grad=True)
sliding_window_attention(Q, K, V, window_size=1).sum().backward()
assert Q.grad is not None

print('All tests passed!')